In [1]:

!pip -q install gradio plotly pandas numpy scikit-learn


In [2]:

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
months = pd.date_range("2025-01-01", periods=24, freq="MS")
rows = []

for i, month in enumerate(months):
    marketing = rng.normal(55, 7)
    conversion = np.clip(3.2 + (marketing - 55) * .025 + rng.normal(0, .25), 2, 5)
    sales_productivity = rng.normal(75, 5)
    delivery = np.clip(rng.normal(92, 3), 80, 99)
    inventory = np.clip(rng.normal(94, 2), 85, 100)
    satisfaction = np.clip(78 + .3*(delivery-90) + rng.normal(0, 2), 60, 100)
    absence = np.clip(rng.normal(4.8, .8), 2, 9)

    revenue = max(400, 900 + marketing*4 + conversion*25
                  + sales_productivity*5 + satisfaction*3
                  + rng.normal(0, 40))
    cost = max(200, 400 + marketing*1.5 + (100-delivery)*4
               + absence*5 + rng.normal(0, 20))

    if i in [7, 15]:
        delivery -= 7
        satisfaction -= 5
        cost += 35

    profit = revenue - cost

    rows.append({
        "Month": month,
        "Revenue_k": round(revenue, 2),
        "Profit_k": round(profit, 2),
        "Operating_Cost_k": round(cost, 2),
        "Profit_Margin_pct": round(profit / revenue * 100, 2),
        "Marketing_Spend_k": round(marketing, 2),
        "Campaign_Conversion_pct": round(conversion, 2),
        "Sales_Productivity_Index": round(sales_productivity, 2),
        "On_Time_Delivery_pct": round(delivery, 2),
        "Inventory_Availability_pct": round(inventory, 2),
        "Customer_Satisfaction_score": round(satisfaction, 2),
        "Employee_Absence_pct": round(absence, 2)
    })

df = pd.DataFrame(rows)
df.head()


,Month,Revenue_k,Profit_k,Operating_Cost_k,Profit_Margin_pct,Marketing_Spend_k,Campaign_Conversion_pct,Sales_Productivity_Index,On_Time_Delivery_pct,Inventory_Availability_pct,Customer_Satisfaction_score,Employee_Absence_pct
0,2025-01-01,1815.00,1284.42,530.59,70.77,57.13,2.99,78.75,94.82,90.10,76.84,4.90
1,2025-02-01,1825.86,1319.72,506.14,72.28,49.03,3.27,78.89,92.20,96.25,79.59,4.11
2,2025-03-01,1817.46,1252.63,564.83,68.92,61.15,3.34,74.08,89.96,96.45,77.68,4.46
3,2025-04-01,1863.33,1327.37,535.96,71.24,57.56,3.37,77.15,98.42,93.19,79.50,4.15
4,2025-05-01,1795.55,1266.38,529.17,70.53,54.20,2.97,70.88,93.95,95.49,80.27,4.27


In [3]:

def analyze_executive_performance(data, target="Profit_k"):
    data = data.copy()
    data["Month"] = pd.to_datetime(data["Month"], errors="coerce")
    data = data.dropna(subset=["Month"]).sort_values("Month")

    if len(data) < 4:
        raise ValueError("At least four monthly records are required.")
    if target not in data.columns:
        raise ValueError(f"Missing financial target: {target}")

    numeric_cols = data.select_dtypes(include="number").columns
    results = []

    for col in numeric_cols:
        if col == target or data[col].nunique() < 2:
            continue

        pair = data[[col, target]].dropna()
        if len(pair) >= 4 and pair[col].nunique() > 1:
            corr = pair[col].corr(pair[target])
            if pd.notna(corr):
                results.append({
                    "KPI": col,
                    "Correlation with outcome": round(corr, 3),
                    "Strength": abs(corr)
                })

    ranking = pd.DataFrame(results)
    if not ranking.empty:
        ranking = ranking.sort_values("Strength", ascending=False)

    latest = data.iloc[-1]
    previous = data.iloc[-2]
    change = (
        (latest[target] - previous[target]) / abs(previous[target]) * 100
        if previous[target] != 0 else 0
    )

    direction = "increased" if change >= 0 else "decreased"
    insights = [
        f"{target} {direction} by {abs(change):.1f}% "
        f"compared with the previous month."
    ]

    if not ranking.empty:
        for _, row in ranking.head(3).iterrows():
            relationship = (
                "positive" if row["Correlation with outcome"] > 0
                else "negative"
            )
            insights.append(
                f"{row['KPI']} has a {relationship} association with "
                f"{target} (correlation: "
                f"{row['Correlation with outcome']:.2f})."
            )

    # Rule-based operational alerts
    if "On_Time_Delivery_pct" in data.columns:
        if latest["On_Time_Delivery_pct"] < 90:
            insights.append(
                "Operations alert: on-time delivery is below 90%. "
                "Investigate routes, carriers and warehouse handoffs."
            )

    if "Customer_Satisfaction_score" in data.columns:
        if latest["Customer_Satisfaction_score"] < 75:
            insights.append(
                "Customer alert: satisfaction is low. Review complaints "
                "and segment the issues before choosing corrective actions."
            )

    if "Employee_Absence_pct" in data.columns:
        if latest["Employee_Absence_pct"] > 6:
            insights.append(
                "Workforce alert: absence exceeds 6%. Review team-level "
                "patterns and staffing coverage."
            )

    insights.append(
        "Decision safeguard: correlation does not prove causation. "
        "Validate these signals with departmental owners before acting."
    )

    return ranking, insights


In [ ]:

import gradio as gr
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Premium executive command-centre theme
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&family=Space+Grotesk:wght@400;500;600;700&display=swap');

:root {
  --bg: #101410;
  --panel: #1B211A;
  --line: #343C31;
  --lime: #D6FF62;
  --text: #F1F4EA;
  --muted: #A5B09D;
}
.gradio-container {
  max-width: 1550px !important;
  background: var(--bg) !important;
  color: var(--text) !important;
  font-family: 'DM Sans', sans-serif !important;
}
body, .main, .wrap { background: var(--bg) !important; }
h1, h2, h3, h4 {
  font-family: 'Space Grotesk', sans-serif !important;
  color: var(--text) !important;
}
.hero {
  padding: 26px;
  border-radius: 18px;
  border: 1px solid var(--line);
  background: linear-gradient(120deg, #20291D, #191E18 60%, #24291A);
  margin-bottom: 18px;
}
.eyebrow {
  color: var(--lime);
  font-size: 11px;
  font-weight: 700;
  letter-spacing: 2px;
}
.hero h1 { font-size: 32px; margin: 10px 0; }
.hero p { color: var(--muted); line-height: 1.7; }
.kpis {
  display: grid;
  grid-template-columns: repeat(4, minmax(0, 1fr));
  gap: 12px;
  margin: 16px 0;
}
.kpi {
  background: var(--panel);
  border: 1px solid var(--line);
  border-radius: 15px;
  padding: 20px;
  min-height: 112px;
}
.kpi span {
  color: var(--muted);
  font-size: 12px;
  display: block;
}
.kpi strong {
  display: block;
  font: 700 27px 'Space Grotesk', sans-serif;
  margin: 10px 0 5px;
  color: var(--text);
}
.kpi small { color: var(--lime); font-size: 11px; }
.panel {
  background: var(--panel);
  border: 1px solid var(--line);
  border-radius: 15px;
  padding: 20px;
  margin: 12px 0;
}
.panel h3 { margin-top: 0; font-size: 17px; }
.brief-list { color: #DCE3D4; line-height: 1.8; }
.brief-list li { margin-bottom: 10px; }
.lime { color: var(--lime); }
.note { color: var(--muted); font-size: 12px; line-height: 1.7; }
button.primary {
  background: var(--lime) !important;
  color: #11160D !important;
  border: none !important;
  font-weight: 700 !important;
}
button { border-radius: 10px !important; }
input, textarea, select {
  border-radius: 9px !important;
}
footer { display: none !important; }
@media(max-width: 850px) {
  .kpis { grid-template-columns: repeat(2, minmax(0, 1fr)); }
}
"""

def premium_dashboard(upload, target):
    # Use the existing synthetic dataset unless a CSV is uploaded.
    data = pd.read_csv(upload) if upload else df.copy()

    ranking, insights = analyze_executive_performance(data, target)

    data["Month"] = pd.to_datetime(data["Month"], errors="coerce")
    data = data.dropna(subset=["Month"]).sort_values("Month")

    latest = data.iloc[-1]
    previous = data.iloc[-2]

    def money(value):
        return f"${value:,.1f}k" if pd.notna(value) else "N/A"

    def metric(value, suffix=""):
        return f"{value:,.1f}{suffix}" if pd.notna(value) else "N/A"

    target_change = (
        (latest[target] - previous[target]) / abs(previous[target]) * 100
        if previous[target] != 0 else 0
    )
    change_color = "#D6FF62" if target_change >= 0 else "#FF937E"
    change_label = "▲" if target_change >= 0 else "▼"

    revenue = latest.get("Revenue_k", np.nan)
    profit = latest.get("Profit_k", np.nan)
    margin = latest.get("Profit_Margin_pct", np.nan)
    cost = latest.get("Operating_Cost_k", np.nan)

    cards = f"""
    <div class="kpis">
      <div class="kpi">
        <span>REVENUE</span>
        <strong>{money(revenue)}</strong>
        <small>Latest reporting period</small>
      </div>
      <div class="kpi">
        <span>NET PROFIT</span>
        <strong>{money(profit)}</strong>
        <small style="color:{change_color}">
          {change_label} {abs(target_change):.1f}% change in selected outcome
        </small>
      </div>
      <div class="kpi">
        <span>PROFIT MARGIN</span>
        <strong>{metric(margin, '%')}</strong>
        <small>Profit / revenue</small>
      </div>
      <div class="kpi">
        <span>OPERATING COST</span>
        <strong>{money(cost)}</strong>
        <small>Latest reporting period</small>
      </div>
    </div>
    <div class="note">
      Reporting period: {latest['Month'].strftime('%B %Y')}
      · Financial values are in thousands ($k) where indicated.
    </div>
    """

    # Executive brief: present findings as a readable report.
    list_items = "".join(f"<li>{item}</li>" for item in insights)
    brief_html = f"""
    <div class="panel">
      <div class="eyebrow">MANAGEMENT BRIEF · {latest['Month'].strftime('%b %Y').upper()}</div>
      <h2 style="margin-top:10px">
        {target.replace('_', ' ')}:
        <span class="lime">{change_label} {abs(target_change):.1f}%</span>
      </h2>
      <p class="note">
        Comparison with the previous reporting period. Signals require
        validation and do not independently establish cause and effect.
      </p>
    </div>
    <div class="panel">
      <h3>What the numbers suggest</h3>
      <ul class="brief-list">{list_items}</ul>
    </div>
    <div class="panel">
      <h3>Recommended executive response</h3>
      <ol class="brief-list">
        <li><b>Validate:</b> ask the relevant department owner to confirm the KPI movement and data quality.</li>
        <li><b>Prioritize:</b> focus on the strongest material signals connected with the selected financial outcome.</li>
        <li><b>Act:</b> assign an owner, a measurable target and a review date to each approved intervention.</li>
      </ol>
      <p class="note">
        This prototype uses correlation screening and rule-based alerts.
        Correlation does not prove causation.
      </p>
    </div>
    """

    # Plotly chart styling
    chart_layout = dict(
        template="plotly_dark",
        paper_bgcolor="#1B211A",
        plot_bgcolor="#1B211A",
        font=dict(family="DM Sans", color="#F1F4EA"),
        title_font=dict(family="Space Grotesk", size=18, color="#F1F4EA"),
        margin=dict(l=30, r=25, t=65, b=35),
        xaxis=dict(gridcolor="#343C31", zerolinecolor="#343C31"),
        yaxis=dict(gridcolor="#343C31", zerolinecolor="#343C31")
    )

    trend = go.Figure()
    trend.add_trace(go.Scatter(
        x=data["Month"], y=data[target],
        mode="lines+markers",
        line=dict(color="#D6FF62", width=3),
        marker=dict(size=7, color="#D6FF62"),
        fill="tozeroy",
        fillcolor="rgba(214,255,98,0.08)",
        name=target.replace("_", " ")
    ))
    trend.update_layout(
        title=f"{target.replace('_', ' ')} · monthly trend",
        **chart_layout
    )

    if ranking.empty:
        driver_chart = go.Figure()
        driver_chart.add_annotation(
            text="No usable numeric KPI relationships found",
            showarrow=False
        )
    else:
        top = ranking.head(8).sort_values("Correlation with outcome")
        bar_colors = [
            "#D6FF62" if value >= 0 else "#FF937E"
            for value in top["Correlation with outcome"]
        ]
        driver_chart = go.Figure(go.Bar(
            x=top["Correlation with outcome"],
            y=top["KPI"].str.replace("_", " "),
            orientation="h",
            marker_color=bar_colors,
            text=top["Correlation with outcome"].map(lambda x: f"{x:+.2f}"),
            textposition="outside"
        ))
        driver_chart.add_vline(x=0, line_color="#899483", line_width=1)
    driver_chart.update_layout(
        title=f"KPI association ranking · {target.replace('_', ' ')}",
        xaxis_title="Correlation (-1 to +1)",
        **chart_layout
    )
    driver_chart.update_xaxes(range=[-1.15, 1.15])

    # Department-level comparisons when department data is present.
    if "Department" in data.columns:
        department_cols = [
            c for c in ["Department", target] if c in data.columns
        ]
        department_data = data[department_cols].groupby(
            "Department", as_index=False
        )[target].mean()
        department_chart = px.bar(
            department_data, x="Department", y=target,
            title="Average outcome by department",
            color=target,
            color_continuous_scale=["#52604A", "#D6FF62"]
        )
        department_chart.update_layout(**chart_layout)
    else:
        department_chart = go.Figure()
        department_chart.add_annotation(
            text="Add a Department column to compare departments.",
            showarrow=False
        )
        department_chart.update_layout(
            title="Department comparison", **chart_layout
        )

    recent = data.tail(12).copy()
    recent["Month"] = recent["Month"].dt.strftime("%b %Y")

    return (
        cards, brief_html, trend, driver_chart,
        department_chart, recent,
        ranking.drop(columns=["Strength"], errors="ignore")
    )

with gr.Blocks(css=custom_css, theme=gr.themes.Base(), title="Executive Intelligence") as app:
    gr.HTML("""
    <div class="hero">
      <div class="eyebrow">STRATEGY · PERFORMANCE · ACTION</div>
      <h1>Executive Decision Intelligence</h1>
      <p>
        Turn KPI overload into a clear business picture.
        Connect operational signals to financial outcomes,
        uncover potential performance drivers, and prioritize
        the next management decision.
      </p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1, min_width=240):
            gr.Markdown("## Control room")
            upload = gr.File(
                label="Upload KPI dataset (CSV)",
                file_types=[".csv"],
                type="filepath"
            )
            target = gr.Dropdown(
                choices=[
                    "Profit_k", "Revenue_k",
                    "Operating_Cost_k", "Profit_Margin_pct"
                ],
                value="Profit_k",
                label="Financial outcome"
            )
            run = gr.Button("✦  Analyse performance", variant="primary")
            gr.Markdown("""
            <div class="panel">
              <div class="eyebrow">HOW IT WORKS</div>
              <p class="note">
                01 · Compare financial performance<br>
                02 · Rank KPI associations<br>
                03 · Detect operational signals<br>
                04 · Generate executive actions
              </p>
            </div>
            <div class="note">
              No upload? The dashboard uses synthetic demonstration data.
              CSV files need a Month column and numeric KPI columns.
            </div>
            """)

        with gr.Column(scale=4):
            cards_out = gr.HTML()
            with gr.Tabs():
                with gr.Tab("✦ Executive Brief"):
                    brief_out = gr.HTML()
                with gr.Tab("01 · Financial Trend"):
                    trend_out = gr.Plot()
                with gr.Tab("02 · KPI Drivers"):
                    driver_out = gr.Plot()
                    ranking_out = gr.Dataframe(
                        label="KPI association details",
                        interactive=False
                    )
                with gr.Tab("03 · Department View"):
                    department_out = gr.Plot()
                with gr.Tab("04 · Data Explorer"):
                    data_out = gr.Dataframe(interactive=False)

    outputs = [
        cards_out, brief_out, trend_out, driver_out,
        department_out, data_out, ranking_out
    ]
    run.click(
        fn=premium_dashboard,
        inputs=[upload, target],
        outputs=outputs
    )
    app.load(
        fn=premium_dashboard,
        inputs=[upload, target],
        outputs=outputs
    )

app.launch(share=True, debug=True)


/tmp/ipykernel_5232/1892624252.py:280: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.



Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2ea2f00cd0c3999cc4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
